In [11]:
# =============================================================================
# PARCIAL 1 PROCESAMIENTO DE DATOS CON PYSPARK
# Estudiante(s): Juan Diego Valencia Alomia - Federico Terán Pascuas
# Correo(s) Institucional: judival30@javerianacali.edu.co - federico04@javerianacali.edu.co
# Fecha y Hora: [Fecha y Hora de Entrega]
# =============================================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
import os


In [12]:
# -----------------------------------------------------------------------------
# TEMA 1: CONFIGURACIÓN Y INGESTION DE DATOS (10%)
# -----------------------------------------------------------------------------
# 1.1 Inicializar SparkSession optimizada para VM (1.5 GB RAM, 4 shuffle partitions)
spark = SparkSession.builder \
.appName("Parcial_PySpark_Estudiante") \
.config("spark.driver.memory", "1536m") \
.config("spark.executor.memory", "1536m") \
.config("spark.sql.shuffle.partitions", "4") \
.getOrCreate()

# 1.2 TODO: Definir el esquema explícito StructType para 'sales_transactions.csv'
# Columnas:
# transaction_id (int),
# user_id (int),
# amount (double),
# transaction_date (string),
# country (string)
schema_transactions = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_date", StringType(), True),
    StructField("country", StringType(), True)
])


#a = os.getcwd()
#print(a)

# 1.3 TODO: Leer el archivo CSV aplicando el esquema explícito y encabezado
path_csv = "/home/fede/project/notebooks/parcial1/sales_transactions.csv"
df_raw = spark.read.csv(path_csv, header=True, schema=schema_transactions)

df_raw.show()

+--------------+-------+------+----------------+-------+
|transaction_id|user_id|amount|transaction_date|country|
+--------------+-------+------+----------------+-------+
|          1001|    101| 150.5|      2025-09-01|     ES|
|          1002|    102|  45.0|      2025-09-01|     ES|
|          1003|    101| 200.0|      2025-09-02|     FR|
|          1004|    103|  NULL|      2025-09-02|     US|
|          1005|    104|  80.2|      2025-09-03|     ES|
|          1006|    105| 310.0|      2025-09-03|     FR|
|          1007|   NULL| 120.0|      2025-09-04|     US|
|          1008|    102|  15.5|      2025-09-04|     ES|
|          1009|    106| 500.0|      2025-09-05|     US|
|          1010|    101|  95.0|      2025-09-05|     ES|
+--------------+-------+------+----------------+-------+



In [16]:
# -----------------------------------------------------------------------------
# TEMA 2: TRANSFORMACIONES Y LIMPIEZA DE DATOS (DataFrame API) (20%)
# -----------------------------------------------------------------------------
# 2.1 TODO: Eliminar registros que contengan nulos en transaction_id, user_id o amount
df_clean = df_raw.dropna(subset=["transaction_id", "user_id", "amount"])

df_clean.show()

# 2.2 TODO: Convertir 'transaction_date' a DateType con formato 'yyyy-MM-dd'
# y crear la columna 'amount_usd' multiplicando 'amount' por 1.10 redondeado a 2 decimales
df_transformed = df_clean \
    .withColumn("transaction_date", F.to_date(F.col("transaction_date"), "yyyy-MM-dd")) \
    .withColumn("amount_usd", F.round(F.col("amount") * 1.10, 2))

df_transformed.show()

# 2.3 TODO: Filtrar únicamente las transacciones con 'amount_usd' estrictamente mayor a 50.0
df_filtered = df_transformed.filter(F.col("amount_usd") > 50)
df_filtered.show()
                                
                                    

+--------------+-------+------+----------------+-------+
|transaction_id|user_id|amount|transaction_date|country|
+--------------+-------+------+----------------+-------+
|          1001|    101| 150.5|      2025-09-01|     ES|
|          1002|    102|  45.0|      2025-09-01|     ES|
|          1003|    101| 200.0|      2025-09-02|     FR|
|          1005|    104|  80.2|      2025-09-03|     ES|
|          1006|    105| 310.0|      2025-09-03|     FR|
|          1008|    102|  15.5|      2025-09-04|     ES|
|          1009|    106| 500.0|      2025-09-05|     US|
|          1010|    101|  95.0|      2025-09-05|     ES|
+--------------+-------+------+----------------+-------+

+--------------+-------+------+----------------+-------+----------+
|transaction_id|user_id|amount|transaction_date|country|amount_usd|
+--------------+-------+------+----------------+-------+----------+
|          1001|    101| 150.5|      2025-09-01|     ES|    165.55|
|          1002|    102|  45.0|      2025-0

In [ ]:
# -----------------------------------------------------------------------------
# TEMA 3: VISTAS TEMPORALES Y CONSULTAS SQL (30%)
# -----------------------------------------------------------------------------
# Datasets de Clientes para la unión
customers_data = [
    (101, "Ana Gómez", "Premium"),
    (102, "Luis Martínez", "Standard"),
    (105, "Marta Dubois", "Premium"),
    (106, "John Doe", "Standard")
]
customers_df = spark.createDataFrame(customers_data, ["user_id", "user_name", "tier"])